# Búsqueda de Noticias sobre Alzheimer con The News API

Este notebook utiliza The News API para obtener noticias, artículos y contenido sobre Alzheimer y demencia desde miles de fuentes de noticias.


## Instalación de dependencias

Instalamos las bibliotecas necesarias para trabajar con The News API y procesar contenido web.


In [94]:
# Instalar dependencias si no están instaladas
!pip install requests beautifulsoup4 lxml
!pip install cloudscraper
!pip install selenium

## Importación de librerías


In [95]:
import requests
from typing import List, Dict, Optional
from datetime import datetime, timedelta
import json
from bs4 import BeautifulSoup
import time
import re
import random

# Intentar importar cloudscraper (mejor para evitar protecciones anti-bot)
try:
    import cloudscraper
    CLOUDSCRAPER_AVAILABLE = True
except ImportError:
    CLOUDSCRAPER_AVAILABLE = False
    print("⚠ cloudscraper no está instalado. Instálalo con: pip install cloudscraper")
    print("  Esto ayudará a evitar errores 403 en sitios protegidos.")

# Intentar importar Playwright (para sitios que requieren JavaScript)
try:
    from playwright.sync_api import sync_playwright
    PLAYWRIGHT_AVAILABLE = True
except ImportError:
    PLAYWRIGHT_AVAILABLE = False
    print("⚠ playwright no está instalado. Instálalo con: pip install playwright")
    print("  Luego ejecuta: playwright install chromium")
    print("  Esto permite renderizar JavaScript en sitios que lo requieren.")

# Intentar importar Selenium (alternativa a Playwright)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False
    print("⚠ selenium no está instalado. Instálalo con: pip install selenium")
    print("  También necesitarás chromedriver o geckodriver instalado.")


## Configuración de The News API

Configura tu API token de The News API. Puedes obtenerlo registrándote gratis en https://www.thenewsapi.com/


### Verificar instalación de cloudscraper

Verifica que cloudscraper esté instalado y funcionando correctamente.


In [96]:
try:
    import cloudscraper
    print("✓ cloudscraper está instalado y disponible")
    print(f"  Versión: {cloudscraper.__version__ if hasattr(cloudscraper, '__version__') else 'desconocida'}")
except ImportError:
    print("⚠ cloudscraper NO está instalado")
    print("  Instálalo ejecutando: !pip install cloudscraper")

try:
    from playwright.sync_api import sync_playwright
    print("✓ playwright está instalado y disponible")
    print("  Para sitios que requieren JavaScript")
except ImportError:
    print("⚠ playwright NO está instalado")
    print("  Instálalo ejecutando: !pip install playwright")
    print("  Luego: !playwright install chromium")

try:
    from selenium import webdriver
    print("✓ selenium está instalado y disponible")
    print("  Alternativa a Playwright para JavaScript")
except ImportError:
    print("⚠ selenium NO está instalado")
    print("  Instálalo ejecutando: !pip install selenium")
    print("  También necesitarás chromedriver instalado")

print()

try:
    from playwright.sync_api import sync_playwright
    print("✓ playwright está instalado y disponible")
    print("  Puede renderizar JavaScript para sitios que lo requieren")
except ImportError:
    print("⚠ playwright NO está instalado")
    print("  Instálalo ejecutando: !pip install playwright")
    print("  Después ejecuta: playwright install chromium")
    print("  Esto permitirá renderizar JavaScript en sitios protegidos")


✓ cloudscraper está instalado y disponible
  Versión: 1.2.71
✓ playwright está instalado y disponible
  Para sitios que requieren JavaScript
✓ selenium está instalado y disponible
  Alternativa a Playwright para JavaScript

✓ playwright está instalado y disponible
  Puede renderizar JavaScript para sitios que lo requieren


In [97]:
THE_NEWS_API_TOKEN = "#"

BASE_URL = "https://api.thenewsapi.com/v1/news"


## Función auxiliar: Obtener contenido con Playwright

Función auxiliar para obtener contenido usando un navegador headless con JavaScript.


In [98]:
def obtener_contenido_con_playwright(url: str, timeout: int = 30) -> Dict[str, str]:
    """
    Obtiene el contenido de una URL usando Playwright (headless browser con JavaScript).
    
    Parámetros:
    -----------
    url : str
        URL de la noticia
    timeout : int
        Tiempo máximo de espera en segundos (por defecto 30)
    
    Retorna:
    --------
    Dict[str, str]
        Diccionario con el HTML renderizado o error
    """
    resultado = {
        "html_content": None,
        "error": None
    }
    
    try:
        from playwright.sync_api import sync_playwright
    except ImportError:
        resultado["error"] = "Playwright no está instalado. Instálalo con: pip install playwright && playwright install chromium"
        return resultado
    
    try:
        print("  🔄 Usando Playwright (headless browser con JavaScript)...")
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            
            # Configurar headers para parecer más real
            page.set_extra_http_headers({
                "Accept-Language": "es-ES,es;q=0.9,en-US;q=0.8,en;q=0.7",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
            })
            
            page.goto(url, wait_until="networkidle", timeout=timeout * 1000)
            # Esperar un poco más para que cargue el contenido dinámico
            page.wait_for_timeout(3000)
            
            html_content = page.content()
            browser.close()
        
        resultado["html_content"] = html_content
        print("  ✓ Contenido obtenido con Playwright (JavaScript renderizado)")
        return resultado
        
    except Exception as e:
        resultado["error"] = f"Error con Playwright: {str(e)}"
        return resultado


## Método 1: Buscar noticias sobre Alzheimer

Este método busca noticias sobre Alzheimer usando el endpoint "All News" de The News API.


In [99]:
def buscar_noticias_alzheimer(
    api_token: str = THE_NEWS_API_TOKEN,
    search_term: str = "Alzheimer",
    limit: int = 50,
    language: str = "es,en",
    categories: Optional[str] = None,
    exclude_categories: Optional[str] = None,
    published_after: Optional[str] = None,
    published_before: Optional[str] = None,
    domains: Optional[str] = None,
    locale: Optional[str] = None
) -> List[Dict]:
    """
    Busca noticias sobre Alzheimer usando The News API.
    
    Parámetros:
    -----------
    api_token : str
        Token de API de The News API
    search_term : str
        Término de búsqueda (por defecto "Alzheimer")
    limit : int
        Número máximo de resultados (por defecto 50, máximo 100)
    language : str
        Idiomas separados por comas (por defecto "es,en")
    categories : str, opcional
        Categorías separadas por comas (business, tech, health, etc.)
    exclude_categories : str, opcional
        Categorías a excluir separadas por comas
    published_after : str, opcional
        Fecha de inicio en formato Y-m-d (ej: "2024-12-01")
    published_before : str, opcional
        Fecha de fin en formato Y-m-d (ej: "2024-12-31")
    domains : str, opcional
        Dominios específicos separados por comas
    locale : str, opcional
        Códigos de país separados por comas (us, es, etc.)
    
    Retorna:
    --------
    List[Dict]
        Lista de diccionarios con información de las noticias
    """
    try:
        url = f"{BASE_URL}/all"
        
        params = {
            "api_token": api_token,
            "search": search_term,
            "limit": min(limit, 100)  # Máximo 100
        }
        
        if language:
            params["language"] = language
        if categories:
            params["categories"] = categories
        if exclude_categories:
            params["exclude_categories"] = exclude_categories
        if published_after:
            params["published_after"] = published_after
        if published_before:
            params["published_before"] = published_before
        if domains:
            params["domains"] = domains
        if locale:
            params["locale"] = locale
        
        print(f"Buscando noticias sobre: {search_term}")
        print(f"Límite de resultados: {limit}")
        if published_after or published_before:
            print(f"Rango de fechas: {published_after or 'inicio'} - {published_before or 'hoy'}")
        
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if "data" in data:
            noticias = data["data"]
            print(f"✓ Se encontraron {len(noticias)} noticias")
            return noticias
        else:
            print("⚠ No se encontraron noticias")
            return []
        
    except requests.exceptions.RequestException as e:
        print(f"Error al buscar noticias: {e}")
        if hasattr(e.response, 'text'):
            print(f"Respuesta del servidor: {e.response.text}")
        raise
    except Exception as e:
        print(f"Error inesperado: {e}")
        raise


In [109]:
def obtener_contenido_completo(url: str, timeout: int = 30, max_retries: int = 2, usar_javascript: bool = False) -> Dict[str, str]:
    """
    Obtiene el contenido completo de una noticia desde su URL.
    
    Parámetros:
    -----------
    url : str
        URL de la noticia
    timeout : int
        Tiempo máximo de espera en segundos (por defecto 30)
    max_retries : int
        Número máximo de reintentos en caso de error (por defecto 2)
    usar_javascript : bool
        Si es True, intenta usar JavaScript automáticamente si fallan otros métodos (por defecto False, pero se activa automáticamente si falla)
    
    Retorna:
    --------
    Dict[str, str]
        Diccionario con:
        - titulo: Título de la noticia
        - contenido: Contenido completo del artículo
        - autor: Autor del artículo (si está disponible)
        - fecha_publicacion: Fecha de publicación (si está disponible)
        - error: Mensaje de error si hubo algún problema
    """
    resultado = {
        "titulo": "",
        "contenido": "",
        "autor": "",
        "fecha_publicacion": "",
        "error": None
    }
    
    response = None
    
    # Verificar dinámicamente si cloudscraper está disponible
    try:
        import cloudscraper
        cloudscraper_available = True
    except ImportError:
        cloudscraper_available = False
    
    # Intentar usar cloudscraper primero (mejor para evitar protecciones anti-bot)
    if cloudscraper_available:
        try:
            scraper = cloudscraper.create_scraper(
                browser={
                    'browser': 'chrome',
                    'platform': 'windows',
                    'desktop': True
                },
                delay=random.uniform(1, 3)  # Delay aleatorio
            )
            response = scraper.get(url, timeout=timeout)
            if response.status_code == 200:
                print("  ✓ Contenido obtenido con cloudscraper")
            else:
                raise Exception(f"Status code: {response.status_code}")
        except Exception as e:
            print(f"  ⚠ cloudscraper falló ({str(e)}), intentando con requests...")
            response = None
    
    # Si cloudscraper no está disponible o falló, usar requests con headers mejorados
    if response is None or response.status_code != 200:
        # Headers completos que simulan un navegador Chrome real
        user_agents = [
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ]
        
        # Crear sesión para mantener cookies y headers
        session = requests.Session()
        
        # Reintentos con backoff
        for intento in range(max_retries + 1):
            try:
                if intento > 0:
                    # Esperar antes de reintentar (backoff exponencial con jitter)
                    wait_time = (2 ** intento) + random.uniform(0, 1)
                    print(f"  Reintentando en {wait_time:.1f} segundos... (intento {intento + 1}/{max_retries + 1})")
                    time.sleep(wait_time)
                
                # Headers con User-Agent aleatorio
                headers = {
                    "User-Agent": random.choice(user_agents),
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
                    "Accept-Language": "es-ES,es;q=0.9,en-US;q=0.8,en;q=0.7",
                    "Accept-Encoding": "gzip, deflate, br",
                    "DNT": "1",
                    "Connection": "keep-alive",
                    "Upgrade-Insecure-Requests": "1",
                    "Sec-Fetch-Dest": "document",
                    "Sec-Fetch-Mode": "navigate",
                    "Sec-Fetch-Site": "none",
                    "Sec-Fetch-User": "?1",
                    "Cache-Control": "max-age=0",
                    "sec-ch-ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
                    "sec-ch-ua-mobile": "?0",
                    "sec-ch-ua-platform": '"Windows"',
                    "Referer": "https://www.google.com/"
                }
                session.headers.update(headers)
                
                # Delay aleatorio para parecer más humano
                if intento == 0:
                    time.sleep(random.uniform(0.5, 1.5))
                
                response = session.get(url, timeout=timeout, allow_redirects=True)
                
                # Si es 403, intentar con diferentes headers
                if response.status_code == 403:
                    if intento < max_retries:
                        print(f"  ⚠ Error 403, cambiando estrategia...")
                        continue
                    else:
                        resultado["error"] = f"Error 403: Acceso denegado. El sitio puede requerir JavaScript o tener protecciones anti-bot muy estrictas."
                        return resultado
                
                response.raise_for_status()
                break
                
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 403 and intento < max_retries:
                    continue
                resultado["error"] = f"Error HTTP {e.response.status_code}: {str(e)}"
                return resultado
            except requests.exceptions.RequestException as e:
                if intento < max_retries:
                    continue
                resultado["error"] = f"Error al descargar la noticia: {str(e)}"
                return resultado
        
        if response is None or response.status_code != 200:
            # Si falló con requests, intentar con JavaScript si está disponible o si se solicita
            if usar_javascript or True:  # Intentar JavaScript automáticamente si falla
                # Verificar si Playwright o Selenium están disponibles
                try:
                    from playwright.sync_api import sync_playwright
                    playwright_available = True
                except ImportError:
                    playwright_available = False
                
                try:
                    from selenium import webdriver
                    from selenium.webdriver.chrome.options import Options
                    from selenium.webdriver.common.by import By
                    from selenium.webdriver.support.ui import WebDriverWait
                    from selenium.webdriver.support import expected_conditions as EC
                    selenium_available = True
                except ImportError:
                    selenium_available = False
                
                # Intentar con Playwright primero (más rápido y moderno)
                if playwright_available:
                    try:
                        print(f"  Intentando con Playwright (JavaScript)...")
                        with sync_playwright() as p:
                            browser = p.chromium.launch(
                                headless=True,
                                args=['--no-sandbox', '--disable-setuid-sandbox', '--disable-dev-shm-usage']
                            )
                            context = browser.new_context(
                                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                                viewport={'width': 1920, 'height': 1080}
                            )
                            page = context.new_page()
                            
                            # Navegar a la página y esperar a que cargue
                            page.goto(url, wait_until='networkidle', timeout=timeout * 1000)
                            
                            # Esperar un poco más para que JavaScript cargue el contenido
                            page.wait_for_timeout(3000)
                            
                            # Obtener el HTML renderizado
                            html_content = page.content()
                            browser.close()
                            
                            # Crear un objeto response simulado para compatibilidad
                            class MockResponse:
                                def __init__(self, content):
                                    self.content = content.encode('utf-8')
                                    self.status_code = 200
                                    self.text = content
                            
                            response = MockResponse(html_content)
                            print("  ✓ Contenido obtenido con Playwright (JavaScript)")
                    except Exception as e:
                        print(f"  ⚠ Playwright falló: {str(e)}")
                        response = None
                
                # Si Playwright no está disponible o falló, intentar con Selenium
                if (response is None or (hasattr(response, 'status_code') and response.status_code != 200)) and selenium_available:
                    try:
                        print(f"  Intentando con Selenium (JavaScript)...")
                        chrome_options = Options()
                        chrome_options.add_argument('--headless')
                        chrome_options.add_argument('--no-sandbox')
                        chrome_options.add_argument('--disable-dev-shm-usage')
                        chrome_options.add_argument('--disable-gpu')
                        chrome_options.add_argument('--window-size=1920,1080')
                        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
                        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
                        chrome_options.add_experimental_option('useAutomationExtension', False)
                        chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
                        
                        driver = webdriver.Chrome(options=chrome_options)
                        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
                        
                        driver.get(url)
                        
                        # Esperar a que la página cargue
                        WebDriverWait(driver, timeout).until(
                            EC.presence_of_element_located((By.TAG_NAME, "body"))
                        )
                        
                        # Esperar más para JavaScript
                        time.sleep(3)
                        
                        # Obtener el HTML renderizado
                        html_content = driver.page_source
                        driver.quit()
                        
                        # Crear un objeto response simulado
                        class MockResponse:
                            def __init__(self, content):
                                self.content = content.encode('utf-8')
                                self.status_code = 200
                                self.text = content
                        
                        response = MockResponse(html_content)
                        print("  ✓ Contenido obtenido con Selenium (JavaScript)")
                    except Exception as e:
                        print(f"  ⚠ Selenium falló: {str(e)}")
                        if response is None:
                            resultado["error"] = f"No se pudo obtener el contenido. Errores: cloudscraper/requests fallaron. JavaScript (Playwright/Selenium) también falló: {str(e)}"
                            return resultado
            
            # Si después de todo no tenemos respuesta válida
            if response is None or (hasattr(response, 'status_code') and response.status_code != 200):
                resultado["error"] = "No se pudo obtener respuesta del servidor después de intentar todos los métodos disponibles"
                return resultado
    
    try:
        # Parsear HTML (puede venir de requests, cloudscraper, Playwright o Selenium)
        if hasattr(response, 'content'):
            html_content = response.content
        elif hasattr(response, 'text'):
            html_content = response.text.encode('utf-8')
        else:
            html_content = str(response).encode('utf-8')
        
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Extraer título
        titulo = ""
        if soup.find('title'):
            titulo = soup.find('title').get_text(strip=True)
        elif soup.find('h1'):
            titulo = soup.find('h1').get_text(strip=True)
        elif soup.find('meta', property='og:title'):
            titulo = soup.find('meta', property='og:title').get('content', '')
        
        resultado["titulo"] = titulo
        
        # Extraer autor
        autor = ""
        # Buscar en meta tags comunes
        author_meta = soup.find('meta', {'name': 'author'}) or \
                     soup.find('meta', {'property': 'article:author'}) or \
                     soup.find('meta', {'name': 'sailthru.author'})
        if author_meta:
            autor = author_meta.get('content', '')
        
        # Buscar en elementos comunes
        if not autor:
            author_elem = soup.find(class_=lambda x: x and 'author' in x.lower()) or \
                         soup.find('span', class_=lambda x: x and 'author' in x.lower()) or \
                         soup.find('div', class_=lambda x: x and 'author' in x.lower())
            if author_elem:
                autor = author_elem.get_text(strip=True)
        
        resultado["autor"] = autor
        
        # Extraer fecha de publicación
        fecha = ""
        date_meta = soup.find('meta', {'property': 'article:published_time'}) or \
                   soup.find('meta', {'name': 'publish-date'}) or \
                   soup.find('meta', {'name': 'pubdate'}) or \
                   soup.find('time', {'datetime': True})
        if date_meta:
            if date_meta.get('content'):
                fecha = date_meta.get('content')
            elif date_meta.get('datetime'):
                fecha = date_meta.get('datetime')
        
        resultado["fecha_publicacion"] = fecha
        
        # Extraer contenido principal
        contenido = ""
        
        # Intentar encontrar el artículo principal
        article = soup.find('article') or \
                 soup.find('div', class_=lambda x: x and ('article' in x.lower() or 'content' in x.lower() or 'post' in x.lower())) or \
                 soup.find('main') or \
                 soup.find('div', {'id': lambda x: x and ('content' in x.lower() or 'article' in x.lower())})
        
        if article:
            # Remover scripts, estilos y otros elementos no deseados
            for script in article(["script", "style", "nav", "footer", "aside", "header"]):
                script.decompose()
            
            # Obtener todos los párrafos
            parrafos = article.find_all(['p', 'div', 'h2', 'h3', 'h4'])
            contenido = "\n\n".join([p.get_text(strip=True) for p in parrafos if p.get_text(strip=True)])
        else:
            # Fallback: obtener todos los párrafos
            parrafos = soup.find_all('p')
            contenido = "\n\n".join([p.get_text(strip=True) for p in parrafos if p.get_text(strip=True)])
        
        # Limpiar el contenido
        contenido = re.sub(r'\s+', ' ', contenido)
        contenido = contenido.strip()
        
        resultado["contenido"] = contenido
        
        if not contenido:
            resultado["error"] = "No se pudo extraer el contenido del artículo"
        
    except requests.exceptions.RequestException as e:
        resultado["error"] = f"Error al descargar la noticia: {str(e)}"
    except Exception as e:
        resultado["error"] = f"Error al procesar la noticia: {str(e)}"
    
    return resultado


In [110]:
# FUNCIÓN CORREGIDA - Reemplaza la función anterior con esta versión
def obtener_noticias_completas(
    search_term: str = "Alzheimer",
    limit: int = 10,
    language: str = "es,en",
    dias_atras: Optional[int] = None,
    obtener_contenido: bool = True,
    usar_javascript: bool = False
) -> List[Dict]:
    """
    Obtiene noticias sobre Alzheimer con información completa.
    
    Parámetros:
    -----------
    search_term : str
        Término de búsqueda (por defecto "Alzheimer")
    limit : int
        Número máximo de resultados (por defecto 10)
    language : str
        Idiomas separados por comas (por defecto "es,en")
    dias_atras : int, opcional
        Número de días hacia atrás desde hoy
    obtener_contenido : bool
        Si es True, obtiene el contenido completo de cada noticia (por defecto True)
    usar_javascript : bool
        Si es True, fuerza el uso de JavaScript (Playwright/Selenium) desde el inicio.
        Si es False, JavaScript se activa automáticamente solo si fallan otros métodos.
    
    Retorna:
    --------
    List[Dict]
        Lista de diccionarios con información completa de cada noticia
    """
    # Calcular fechas si se especifica días_atras
    published_after = None
    if dias_atras:
        fecha_inicio = datetime.now() - timedelta(days=dias_atras)
        published_after = fecha_inicio.strftime("%Y-%m-%d")
    
    # Buscar noticias
    noticias = buscar_noticias_alzheimer(
        search_term=search_term,
        limit=limit,
        language=language,
        published_after=published_after
    )
    
    if not noticias:
        return []
    
    # Enriquecer con contenido completo si se solicita
    noticias_completas = []
    for i, noticia in enumerate(noticias, 1):
        print(f"\nProcesando noticia {i}/{len(noticias)}: {noticia.get('title', 'Sin título')[:50]}...")
        
        noticia_completa = {
            "uuid": noticia.get("uuid", ""),
            "titulo": noticia.get("title", ""),
            "descripcion": noticia.get("description", ""),
            "snippet": noticia.get("snippet", ""),
            "url": noticia.get("url", ""),
            "image_url": noticia.get("image_url", ""),
            "language": noticia.get("language", ""),
            "published_at": noticia.get("published_at", ""),
            "source": noticia.get("source", ""),
            "categories": noticia.get("categories", []),
            "locale": noticia.get("locale", ""),
            "keywords": noticia.get("keywords", ""),
            "contenido_completo": "",
            "autor": "",
            "fecha_publicacion_original": ""
        }
        
        # Obtener contenido completo si se solicita
        if obtener_contenido and noticia.get("url"):
            try:
                contenido_info = obtener_contenido_completo(
                    noticia["url"],
                    usar_javascript=usar_javascript
                )
                noticia_completa["contenido_completo"] = contenido_info.get("contenido", "")
                noticia_completa["autor"] = contenido_info.get("autor", "") or noticia_completa["autor"]
                noticia_completa["fecha_publicacion_original"] = contenido_info.get("fecha_publicacion", "")
                
                if contenido_info.get("error"):
                    print(f"  ⚠ {contenido_info['error']}")
                    # Guardar el error pero continuar con la siguiente noticia
                    noticia_completa["error_contenido"] = contenido_info.get("error")
                else:
                    print(f"  ✓ Contenido obtenido ({len(noticia_completa['contenido_completo'])} caracteres)")
                
                # Respetar rate limits con delay aleatorio
                time.sleep(random.uniform(0.5, 1.5))
            except Exception as e:
                print(f"  ⚠ Error inesperado al obtener contenido: {e}")
                noticia_completa["error_contenido"] = str(e)
        
        noticias_completas.append(noticia_completa)
    
    # Resumen final
    exitosos = sum(1 for n in noticias_completas if n.get("contenido_completo"))
    fallidos = len(noticias_completas) - exitosos
    print(f"\n{'='*60}")
    print(f"Resumen: {exitosos} noticias con contenido completo, {fallidos} sin contenido")
    print(f"{'='*60}")
    
    return noticias_completas


In [102]:
def obtener_noticias_completas(
    search_term: str = "Alzheimer",
    limit: int = 10,
    language: str = "es,en",
    dias_atras: Optional[int] = None,
    obtener_contenido: bool = True
) -> List[Dict]:
    """
    Obtiene noticias sobre Alzheimer con información completa.
    
    Parámetros:
    -----------
    search_term : str
        Término de búsqueda (por defecto "Alzheimer")
    limit : int
        Número máximo de resultados (por defecto 10)
    language : str
        Idiomas separados por comas (por defecto "es,en")
    dias_atras : int, opcional
        Número de días hacia atrás desde hoy
    obtener_contenido : bool
        Si es True, obtiene el contenido completo de cada noticia (por defecto True)
    
    Retorna:
    --------
    List[Dict]
        Lista de diccionarios con información completa de cada noticia
    """
    # Calcular fechas si se especifica días_atras
    published_after = None
    if dias_atras:
        fecha_inicio = datetime.now() - timedelta(days=dias_atras)
        published_after = fecha_inicio.strftime("%Y-%m-%d")
    
    # Buscar noticias
    noticias = buscar_noticias_alzheimer(
        search_term=search_term,
        limit=limit,
        language=language,
        published_after=published_after
    )
    
    if not noticias:
        return []
    
    # Enriquecer con contenido completo si se solicita
    noticias_completas = []
    for i, noticia in enumerate(noticias, 1):
        print(f"\nProcesando noticia {i}/{len(noticias)}: {noticia.get('title', 'Sin título')[:50]}...")
        
        noticia_completa = {
            "uuid": noticia.get("uuid", ""),
            "titulo": noticia.get("title", ""),
            "descripcion": noticia.get("description", ""),
            "snippet": noticia.get("snippet", ""),
            "url": noticia.get("url", ""),
            "image_url": noticia.get("image_url", ""),
            "language": noticia.get("language", ""),
            "published_at": noticia.get("published_at", ""),
            "source": noticia.get("source", ""),
            "categories": noticia.get("categories", []),
            "locale": noticia.get("locale", ""),
            "keywords": noticia.get("keywords", ""),
            "contenido_completo": "",
            "autor": "",
            "fecha_publicacion_original": ""
        }
        
        # Obtener contenido completo si se solicita
        if obtener_contenido and noticia.get("url"):
            try:
                contenido_info = obtener_contenido_completo(noticia["url"])
                noticia_completa["contenido_completo"] = contenido_info.get("contenido", "")
                noticia_completa["autor"] = contenido_info.get("autor", "") or noticia_completa["autor"]
                noticia_completa["fecha_publicacion_original"] = contenido_info.get("fecha_publicacion", "")
                
                if contenido_info.get("error"):
                    print(f"  ⚠ {contenido_info['error']}")
                    # Guardar el error pero continuar con la siguiente noticia
                    noticia_completa["error_contenido"] = contenido_info.get("error")
                else:
                    print(f"  ✓ Contenido obtenido ({len(noticia_completa['contenido_completo'])} caracteres)")
                
                # Respetar rate limits con delay aleatorio
                time.sleep(random.uniform(0.5, 1.5))
            except Exception as e:
                print(f"  ⚠ Error inesperado al obtener contenido: {e}")
                noticia_completa["error_contenido"] = str(e)
        
        noticias_completas.append(noticia_completa)
    
    # Resumen final
    exitosos = sum(1 for n in noticias_completas if n.get("contenido_completo"))
    fallidos = len(noticias_completas) - exitosos
    print(f"\n{'='*60}")
    print(f"Resumen: {exitosos} noticias con contenido completo, {fallidos} sin contenido")
    print(f"{'='*60}")
    
    return noticias_completas


## Ejemplos de uso

A continuación se muestran ejemplos de cómo usar los métodos para obtener noticias sobre Alzheimer.


### Ejemplo 1: Buscar noticias recientes sobre Alzheimer


In [103]:
fecha_inicio = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

noticias = buscar_noticias_alzheimer(
    search_term="Alzheimer",
    limit=10,
    language="es,en",
    published_after=fecha_inicio
)

print(f"\n=== NOTICIAS ENCONTRADAS ===\n")
for i, noticia in enumerate(noticias[:5], 1):
    print(f"{i}. {noticia.get('title', 'Sin título')}")
    print(f"   Fuente: {noticia.get('source', 'Desconocida')}")
    print(f"   Fecha: {noticia.get('published_at', 'No disponible')}")
    print(f"   Descripción: {noticia.get('description', 'No disponible')[:100]}...")
    print(f"   URL: {noticia.get('url', '')}")
    print()


Buscando noticias sobre: Alzheimer
Límite de resultados: 10
Rango de fechas: 2025-11-30 - hoy
✓ Se encontraron 3 noticias

=== NOTICIAS ENCONTRADAS ===

1. Alzheimer: una pista inesperada revela cómo el cerebro intenta protegerse
   Fuente: europapress.es
   Fecha: 2025-12-01T06:49:48.000000Z
   Descripción: Utilizando modelos de ratón con Alzheimer, células humanas y tejido cerebral humano, investigadores ...
   URL: https://www.infosalus.com/mayores/noticia-alzheimer-pista-inesperada-revela-cerebro-intenta-protegerse-20251201074948.html

2. CEAFA pide en el Congreso actualizar el Plan Integral de Alzheimer y Otras Demencias
   Fuente: europapress.es
   Fecha: 2025-12-05T15:53:17.000000Z
   Descripción: La presidenta de la Confederación Española de Alzheimer y otras Demencias (CEAFA), Mariló Almagro, h...
   URL: https://www.infosalus.com/asistencia/noticia-ceafa-pide-congreso-actualizar-plan-integral-alzheimer-otras-demencias-20251205165317.html

3. La ciencia señala los pasos diario

### Ejemplo 2: Obtener noticias con contenido completo


In [114]:
noticias_completas = obtener_noticias_completas(
    search_term="Alzheimer OR dementia",
    limit=5,
    language="es,en",
    dias_atras=30,
    obtener_contenido=True
)

if noticias_completas:
    primera_noticia = noticias_completas[0]
    print(f"\n=== NOTICIA COMPLETA ===\n")
    print(f"Título: {primera_noticia['titulo']}")
    print(f"Fuente: {primera_noticia['source']}")
    print(f"Fecha: {primera_noticia['published_at']}")
    print(f"Autor: {primera_noticia['autor'] if primera_noticia['autor'] else 'No disponible'}")
    print(f"\nDescripción: {primera_noticia['descripcion']}")
    print(f"\nSnippet: {primera_noticia['snippet']}")
    
    if primera_noticia['contenido_completo']:
        print(f"\n=== CONTENIDO COMPLETO ===")
        print(f"Longitud: {len(primera_noticia['contenido_completo'])} caracteres")
        print(f"\nPrimeros 1000 caracteres:\n{primera_noticia['contenido_completo'][:1000]}...")
    else:
        print("\n⚠ No se pudo obtener el contenido completo")


Buscando noticias sobre: Alzheimer OR dementia
Límite de resultados: 5
Rango de fechas: 2025-11-07 - hoy
✓ Se encontraron 3 noticias

Procesando noticia 1/3: Dementia Risk Intertwined With Exercise at Two Lif...
  ⚠ cloudscraper falló (Status code: 403), intentando con requests...
  ⚠ Error 403, cambiando estrategia...
  Reintentando en 2.3 segundos... (intento 2/3)
  ⚠ Error 403, cambiando estrategia...
  Reintentando en 4.6 segundos... (intento 3/3)
  ⚠ Error 403: Acceso denegado. El sitio puede requerir JavaScript o tener protecciones anti-bot muy estrictas.

Procesando noticia 2/3: Rising LATE Dementia Diagnoses; Wooden Neurons; Re...
  ⚠ cloudscraper falló (Status code: 403), intentando con requests...
  ⚠ Error 403, cambiando estrategia...
  Reintentando en 2.6 segundos... (intento 2/3)
  ⚠ Error 403, cambiando estrategia...
  Reintentando en 5.0 segundos... (intento 3/3)
  ⚠ Error 403: Acceso denegado. El sitio puede requerir JavaScript o tener protecciones anti-bot muy estricta

### Ejemplo 3: Buscar noticias con filtros específicos


In [105]:
noticias_filtradas = buscar_noticias_alzheimer(
    search_term="Alzheimer treatment OR Alzheimer research",
    limit=20,
    language="en",
    categories="health",
    exclude_categories="sports,entertainment",
    published_after="2024-11-01",
    locale="us,es,gb"
)

print(f"\n=== NOTICIAS FILTRADAS ===\n")
print(f"Total encontradas: {len(noticias_filtradas)}\n")

for noticia in noticias_filtradas[:3]:
    print(f"• {noticia.get('title', 'Sin título')}")
    print(f"  Categorías: {', '.join(noticia.get('categories', []))}")
    print(f"  Locale: {noticia.get('locale', 'N/A')}")
    print()


Buscando noticias sobre: Alzheimer treatment OR Alzheimer research
Límite de resultados: 20
Rango de fechas: 2024-11-01 - hoy
✓ Se encontraron 3 noticias

=== NOTICIAS FILTRADAS ===

Total encontradas: 3

• Alzheimer Disease in Breast Cancer Survivors
  Categorías: health
  Locale: N/A

• Addressing Digital Disparities in Alzheimer Disease by Improving Access to Alzheimer Resources for Spanish-Speaking Latino or Latina Individuals in Los Angeles County: Mixed Methods Study
  Categorías: health, science
  Locale: N/A

• Digital Tools for People Without an Alzheimer Disease or Dementia Diagnosis: Scoping Review
  Categorías: health, science
  Locale: N/A



### Ejemplo 5: Forzar uso de JavaScript para sitios protegidos


In [113]:
url_protegida = "https://www.medpagetoday.com/neurology/dementia/118611"

print("Intentando obtener contenido de un sitio protegido...")
contenido_js = obtener_contenido_completo(
    url=url_protegida,
    timeout=30,
    usar_javascript=True
)

if contenido_js.get("error"):
    print(f"Error: {contenido_js['error']}")
else:
    print(f"✓ Título: {contenido_js['titulo']}")
    print(f"✓ Autor: {contenido_js['autor'] if contenido_js['autor'] else 'No disponible'}")
    print(f"✓ Contenido: {len(contenido_js['contenido'])} caracteres")
    if contenido_js['contenido']:
        print(f"\nPrimeros 500 caracteres:\n{contenido_js['contenido'][:500]}...")


Intentando obtener contenido de un sitio protegido...
  ⚠ cloudscraper falló (Status code: 403), intentando con requests...
  ⚠ Error 403, cambiando estrategia...
  Reintentando en 2.8 segundos... (intento 2/3)
  ⚠ Error 403, cambiando estrategia...
  Reintentando en 4.2 segundos... (intento 3/3)
Error: Error 403: Acceso denegado. El sitio puede requerir JavaScript o tener protecciones anti-bot muy estrictas.


### Ejemplo 4: Exportar noticias a JSON


In [112]:
noticias_export = obtener_noticias_completas(
    search_term="Alzheimer",
    limit=10,
    language="es,en",
    dias_atras=7,
    obtener_contenido=True,
    usar_javascript=False  # Cambiar a True para sitios que requieren JavaScript
)

with open("noticias_alzheimer.json", "w", encoding="utf-8") as f:
    json.dump(noticias_export, f, indent=2, ensure_ascii=False, default=str)

print(f"✓ Se exportaron {len(noticias_export)} noticias a noticias_alzheimer.json")

for noticia in noticias_export[:3]:
    print(f"\n• {noticia['titulo']}")
    print(f"  Fuente: {noticia['source']}")
    print(f"  Fecha: {noticia['published_at']}")
    if noticia['contenido_completo']:
        print(f"  Contenido: {len(noticia['contenido_completo'])} caracteres")


Buscando noticias sobre: Alzheimer
Límite de resultados: 10
Rango de fechas: 2025-11-30 - hoy
✓ Se encontraron 3 noticias

Procesando noticia 1/3: Alzheimer: una pista inesperada revela cómo el cer...
  ✓ Contenido obtenido con cloudscraper
  ✓ Contenido obtenido (17 caracteres)

Procesando noticia 2/3: CEAFA pide en el Congreso actualizar el Plan Integ...
  ✓ Contenido obtenido con cloudscraper
  ✓ Contenido obtenido (20 caracteres)

Procesando noticia 3/3: La ciencia señala los pasos diarios que podrían re...
  ✓ Contenido obtenido con cloudscraper
  ✓ Contenido obtenido (17 caracteres)

Resumen: 3 noticias con contenido completo, 0 sin contenido
✓ Se exportaron 3 noticias a noticias_alzheimer.json

• Alzheimer: una pista inesperada revela cómo el cerebro intenta protegerse
  Fuente: europapress.es
  Fecha: 2025-12-01T06:49:48.000000Z
  Contenido: 17 caracteres

• CEAFA pide en el Congreso actualizar el Plan Integral de Alzheimer y Otras Demencias
  Fuente: europapress.es
  Fecha: 20

## Notas importantes

- **API Token**: Necesitas registrarte gratis en https://www.thenewsapi.com/ para obtener tu API token.

### Librerías recomendadas (en orden de prioridad):

1. **cloudscraper** (`pip install cloudscraper`): 
   - Evita errores 403 en sitios protegidos
   - Rápido y eficiente
   - Se usa automáticamente si está instalado

2. **Playwright** (`pip install playwright` y luego `playwright install chromium`):
   - Renderiza JavaScript completo
   - Funciona en servidores (headless)
   - Más rápido que Selenium
   - **Se activa automáticamente** si cloudscraper/requests fallan

3. **Selenium** (`pip install selenium` + chromedriver):
   - Alternativa a Playwright
   - También renderiza JavaScript
   - Requiere chromedriver instalado
   - **Se activa automáticamente** si Playwright no está disponible

### Estrategia automática de obtención de contenido:

La función intenta en este orden (automáticamente):
1. **cloudscraper** (si está disponible) - más rápido
2. **requests** con headers mejorados (si cloudscraper falla)
3. **Playwright** con JavaScript (si todo lo anterior falla) - **automático**
4. **Selenium** con JavaScript (si Playwright no está disponible) - **automático**

### Para servidores:

- **Playwright** es la mejor opción: más rápido, más estable, funciona perfectamente en servidores Linux sin interfaz gráfica
- **Selenium** también funciona pero requiere más configuración (chromedriver)
- Ambos funcionan en modo headless (sin interfaz gráfica)
- Playwright incluye Chromium, no necesitas instalarlo por separado

### Otros puntos importantes:

- **Errores 403**: Algunos sitios tienen protecciones muy estrictas. La función intenta automáticamente con JavaScript si fallan otros métodos.
- **Contenido corto**: Si el contenido extraído es muy corto, puede requerir más tiempo de espera para JavaScript o el sitio puede tener estructura no estándar.
- **Rate Limits**: La función incluye delays aleatorios para respetar los límites de los sitios web.
- **Continuidad**: Si una noticia falla, el proceso continúa con las siguientes.
- **Búsqueda avanzada**: Puedes usar operadores: `OR`, `AND`/`+`, `NOT`/`-`, paréntesis.
- **Fechas**: Formato Y-m-d (ej: "2024-12-01").
- **Límite máximo**: 100 resultados por petición.


### Ejemplo 5: Usar Playwright para sitios que requieren JavaScript


In [111]:
noticias_js = obtener_noticias_completas(
    search_term="Alzheimer",
    limit=3,
    language="es,en",
    dias_atras=30,
    obtener_contenido=True,
    usar_javascript=True
)

if noticias_js:
    for noticia in noticias_js:
        print(f"\n{'='*60}")
        print(f"Título: {noticia['titulo']}")
        print(f"Fuente: {noticia['source']}")
        if noticia.get('error_contenido'):
            print(f"⚠ Error: {noticia['error_contenido']}")
        elif noticia['contenido_completo']:
            print(f"✓ Contenido: {len(noticia['contenido_completo'])} caracteres")
        print(f"{'='*60}")


Buscando noticias sobre: Alzheimer
Límite de resultados: 3
Rango de fechas: 2025-11-07 - hoy
✓ Se encontraron 3 noticias

Procesando noticia 1/3: Alzheimer: una pista inesperada revela cómo el cer...
  ✓ Contenido obtenido con cloudscraper
  ✓ Contenido obtenido (17 caracteres)

Procesando noticia 2/3: El Ayuntamiento aborda la prevención del Alzheimer...
  ✓ Contenido obtenido con cloudscraper
  ✓ Contenido obtenido (8208 caracteres)

Procesando noticia 3/3: Def Con Dos anuncia gira 30 aniversario de Alzheim...
  ✓ Contenido obtenido con cloudscraper
  ✓ Contenido obtenido (1645 caracteres)

Resumen: 3 noticias con contenido completo, 0 sin contenido

Título: Alzheimer: una pista inesperada revela cómo el cerebro intenta protegerse
Fuente: europapress.es
✓ Contenido: 17 caracteres

Título: El Ayuntamiento aborda la prevención del Alzheimer
Fuente: diariopalentino.es
✓ Contenido: 8208 caracteres

Título: Def Con Dos anuncia gira 30 aniversario de Alzheimer para 2026
Fuente: manerasdevi